<h1 style="font-size: 1.6rem; font-weight: bold">ITO5163: Introduction to Cybersecurity</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 3: Advanced Asymmetric Crytosystem</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Identity-Based Cryptography**

In every public-key system so far, **each user generates their own key pair** and publishes the public key. The problem is proving that a public key really belongs to the person it claims to. This needs trusted **Certification Authorities (CAs)**, which issue **certificates** linking public keys to identities. Running and checking this system is costly.

**Identity-Based Cryptography (IBC)** removes the need for certificates. The idea was proposed by **Adi Shamir** in 1984, and the first practical encryption scheme came from **Boneh and Franklin** in 2001.

- **The Core Idea**:
  - Each user has a **unique identity**, such as an email address, name or ID number
  - The **identity itself acts as the public key**, so no separate public key or certificate is needed
  - Users **don't generate their own keys**: a trusted **Private Key Generator (PKG)** issues each user's private key
  - The PKG can be a dedicated server, or an administrator using a central server

#### **How It Works**

```mermaid
sequenceDiagram
    participant PKG as Private Key Generator
    participant A as Alice
    participant B as Bob
    Note over PKG: Setup: master secret s<br>master public key K = g^s
    PKG-->>A: Public parameters (including K)
    PKG-->>B: Public parameters (including K)
    B->>PKG: Proves identity "bob@example.com"
    PKG->>B: Private key dID = H1(ID)^s (secure channel)
    Note over A: Encrypt using only Bob's ID and K<br>c = (u, v)
    A->>B: Ciphertext c
    Note over B: Decrypt with dID
```

- **Overview**:
  - **Setup**: the PKG creates a **master secret key** (kept private) and a **master public key** (published as part of the public parameters)
  - **Key generation**: a user proves their identity to the PKG, which uses the master secret key to create that user's **private key** and sends it securely
  - **Encryption**: the sender only needs the receiver's **identity** and the PKG's **master public key**
  - **Decryption**: the receiver uses the private key the PKG gave them

#### **Bilinear Pairings**

Boneh-Franklin is built on a **Bilinear Pairing**, a special function computed on **elliptic curve** groups. It takes two elements of a group $G$ and maps them to an element of another group $G_T$:

$$e: G \times G \longrightarrow G_T$$

The key property is **bilinearity**: exponents can be moved freely in and out of the pairing.

$$e(g^a, g^b) = e(g, g)^{ab} = e(g^b, g^a)$$

- **Requirements**:
  - **Bilinear**: the property above
  - **Non-degenerate**: $e(g, g) \neq 1$
  - **Efficient**: the pairing is easy to compute
  - **Hard problem**: the discrete logarithm is hard in both $G$ and $G_T$
- The detailed maths of pairings is beyond the scope of this unit

####

### **ElGamal Cryptosystem**

The **ElGamal Cryptosystem**, proposed by **Taher ElGamal** in 1985, is also based on the **Discrete Logarithm Problem**. It works very much like Diffie-Hellman, but Diffie-Hellman only **agrees on a key**, while ElGamal can **encrypt and decrypt messages** directly.

```mermaid
sequenceDiagram
    participant B as Bob (sender)
    participant A as Alice (receiver)
    Note over A: Key generation<br>Private X, public Y = α^X mod q
    A->>B: Public key {q, α, Y}
    Note over B: Pick random k<br>K = Y^k mod q<br>C1 = α^k mod q<br>C2 = K·M mod q
    B->>A: Ciphertext (C1, C2)
    Note over A: K = C1^X mod q<br>M = C2·K⁻¹ mod q
```

#### **Key Generation**

Like Diffie-Hellman, two values are agreed on and made **public**:

- $q$: a large prime number
- $\alpha$: a primitive root of $q$

- **Each User**:
  - Picks a random **private key** $X$, with $1 \le X < q - 1$
  - Computes the **public key** $Y = \alpha^X \bmod q$
  - Shares $\{q, \alpha, Y\}$ with anyone who wants to send them a message

#### **Encryption**

To send a message to Alice, Bob needs Alice's public key $\{q, \alpha, Y_A\}$.

- **Encryption Steps**:
  - Step 1: Represent the message as an integer $0 \le M < q$
    - Longer messages are split into blocks, each smaller than $q$
  - Step 2: Pick a random $k$ with $1 \le k < q - 1$
  - Step 3: Compute a **one-time key** $K = (Y_A)^k \bmod q$
  - Step 4: Encrypt $M$ as a **pair of integers** $(C_1, C_2)$:

$$C_1 = \alpha^k \bmod q \qquad C_2 = KM \bmod q$$

#### **Decryption**

- **Decryption Steps**:
  - Step 1: Recover the one-time key using the private key: $K = (C_1)^{X_A} \bmod q$
  - Step 2: Find $K^{-1} \bmod q$, using the Extended Euclidean Algorithm
  - Step 3: Recover the message: $M = (C_2 K^{-1}) \bmod q$

#### **Why It Works**

Alice can rebuild $K$ from $C_1$ without knowing $k$:

$$K = (Y_A)^k \bmod q = (\alpha^{X_A})^k \bmod q = (\alpha^k)^{X_A} \bmod q = (C_1)^{X_A} \bmod q$$

Once she has $K$, multiplying by $K^{-1}$ cancels it out:

$$C_2 K^{-1} \bmod q = K M K^{-1} \bmod q = M$$

#### **ElGamal Execution for $q = 19$, $\alpha = 10$**

The primitive roots of 19 are $\{2, 3, 10, 13, 14, 15\}$, so $\alpha = 10$ is valid.

| Phase | Who | Working | Result |
|-------|-----|---------|--------|
| Key generation | Alice | Private $X_A = 5$ | |
| | Alice | $Y_A = 10^5 \bmod 19$ | $Y_A = 3$ |
| | Alice | Public key $\{q, \alpha, Y_A\}$ | $\{19, 10, 3\}$ |
| Encryption ($M = 17$) | Bob | Random $k = 6$ | |
| | Bob | $K = 3^6 \bmod 19$ | $K = 7$ |
| | Bob | $C_1 = 10^6 \bmod 19$ | $C_1 = 11$ |
| | Bob | $C_2 = 7 \times 17 \bmod 19 = 119 \bmod 19$ | $C_2 = 5$ |
| Decryption | Alice | $K = 11^5 \bmod 19$ | $K = 7$ |
| | Alice | $7^{-1} \bmod 19$ (since $7 \times 11 = 77 = 4 \times 19 + 1$) | $K^{-1} = 11$ |
| | Alice | $M = 5 \times 11 \bmod 19 = 55 \bmod 19$ | $M = 17$ ✓ |

Final result:
- Ciphertext: $(C_1, C_2) = (11, 5)$
- Decrypted message: $17$

#### **Properties of ElGamal**

- **Ciphertext is twice as long**: each block $M$ becomes two numbers, $C_1$ and $C_2$
- **Randomised**: because $k$ is random, the same message encrypts to a **different ciphertext** every time
- **Never reuse $k$**: every block must use a new $k$
  - With the same $k$, every block gets the same $K$, and $C_1$ is identical, so an attacker can spot the reuse
  - If the attacker knows one plaintext block $M_1$, they get $K = C_2 M_1^{-1} \bmod q$ and can decrypt every other block

| Feature            | Diffie-Hellman              | ElGamal                          |
|--------------------|-----------------------------|----------------------------------|
| Hard problem       | Discrete logarithm          | Discrete logarithm               |
| Purpose            | Agree on a shared key       | Encrypt and decrypt messages     |
| Public values      | $q$, $\alpha$, $Y$          | $q$, $\alpha$, $Y$               |
| Per-message random value | -                     | $k$, new for every block         |
| Output             | A shared key $K$            | Ciphertext pair $(C_1, C_2)$     |



In [1]:
# Toy Boneh-Franklin identity-based encryption, to show how the steps fit together.
# NOT SECURE: a real pairing works on elliptic curve points. Here each element of G is
# stored as its exponent, so discrete logs are visible to everyone. Only the algebra is real.
import hashlib
import secrets

q = 1019                 # prime order of the groups G and G_T
P = 2 * q + 1            # 2039, a prime, so the squares mod P form a group of order q
gT = 4                   # generator of G_T (a square mod P, so it has order q)
g = 1                    # generator of G, written as the exponent 1 ("g^1")


def g_pow(x, k):
    """g^k in G. Elements of G are stored as exponents, so this is multiplication mod q."""
    return (x * k) % q


def pairing(a, b):
    """Toy bilinear map e: G x G -> G_T with e(g^a, g^b) = e(g, g)^(ab)."""
    return pow(gT, (a * b) % q, P)


def H1(identity):
    """Hash an identity string to a non-identity element of G."""
    h = int.from_bytes(hashlib.sha256(identity.encode()).digest(), "big")
    return h % (q - 1) + 1


def H2(t, length):
    """Hash an element of G_T to a mask of `length` bytes."""
    return hashlib.shake_256(str(t).encode()).digest(length)


def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))


# ---------- system setup (Private Key Generator) ----------
s = secrets.randbelow(q - 1) + 1          # master secret key
K = g_pow(g, s)                           # master public key K = g^s

# ---------- key generation (PKG gives Bob his private key) ----------
bob = "bob@example.com"
Q_bob = H1(bob)
d_bob = g_pow(Q_bob, s)                   # d_ID = (Q_ID)^s

# ---------- encryption (Alice only needs Bob's identity and K) ----------
m = b"Hello Bob"
r = secrets.randbelow(q - 1) + 1
T = pairing(H1(bob), K)                   # T_ID = e(Q_ID, K)
u = g_pow(g, r)                           # u = g^r
v = xor(m, H2(pow(T, r, P), len(m)))      # v = m XOR H2(T_ID^r)

# ---------- decryption (Bob uses his private key) ----------
recovered = xor(v, H2(pairing(d_bob, u), len(v)))   # m = v XOR H2(e(d_ID, u))

print("Bob's identity:   ", bob)
print("Ciphertext (u, v):", (u, v.hex()))
print("Decrypted:        ", recovered.decode())
print("e(d_ID, u) == T_ID^r:", pairing(d_bob, u) == pow(T, r, P))

# A different user's private key cannot decrypt Bob's message
d_eve = g_pow(H1("eve@example.com"), s)
print("Eve's key gives:  ", xor(v, H2(pairing(d_eve, u), len(v))))

Bob's identity:    bob@example.com
Ciphertext (u, v): (753, '01efc49b1db9cff424')
Decrypted:         Hello Bob
e(d_ID, u) == T_ID^r: True
Eve's key gives:   b'\xc5#N\xbd\xef#\xbb\xf8\xc4'


<br>

***

### **Attribute-Based Cryptography**

**Identity-based cryptography** removes the need for certificates, but it's still **one-to-one**: each ciphertext is encrypted for **one specific identity**. To send the same message to many people, the sender must encrypt it separately for each one, and must know exactly who they are.

**Attribute-Based Cryptography (ABC)**, introduced by **Sahai and Waters** in 2005, solves this. Instead of a single identity, each user is described by a **set of attributes**, and a message can be encrypted for **anyone whose attributes match a policy**.

- **Example Attribute Sets**:
  - {male, student, computer science, Monash University}
  - {female, accountant, Company ABC}
  - {Japanese, born 1980-1989, PhD}

#### **How It Works**

A trusted **Attribute Authority** gives each user a private key based on their attributes. The sender encrypts the message under an **Access Policy**: a rule built from attributes with **AND** and **OR**. Only users whose attributes **satisfy the policy** can decrypt.

```mermaid
flowchart LR
    S["Sender"] -- "encrypts with policy<br>student AND computer science<br>AND Monash" --> C[("Ciphertext")]
    C --> U1["User 1<br>{student, computer science, Monash}"]
    C --> U2["User 2<br>{student, math, Monash}"]
    C --> U3["User 3<br>{lecturer, computer science, Monash}"]
    U1 --> R1["✓ Can decrypt"]
    U2 --> R2["✗ Cannot decrypt"]
    U3 --> R3["✗ Cannot decrypt"]
```

| Policy in Words | Policy |
|-----------------|--------|
| Computer Science students at Monash | {student AND computer science AND Monash} |
| Male Japanese or female Koreans | {(male AND Japanese) OR (female AND Korean)} |

- **Steps**:
  - **Setup**: the Attribute Authority creates a master secret key and public parameters
  - **Key generation**: each user gets a private key tied to **their attributes**
  - **Encryption**: the sender encrypts under a **policy**, without needing to know who the receivers are
  - **Decryption**: works only if the user's attributes **satisfy** the policy

#### **Two Types of ABE**

| Type | Policy Attached To | Attributes Attached To | Who Decides Access |
|------|--------------------|------------------------|--------------------|
| **Ciphertext-Policy ABE (CP-ABE)** | The ciphertext | The user's key | The sender (the examples above) |
| **Key-Policy ABE (KP-ABE)** | The user's key | The ciphertext | The authority |

#### **Collusion Resistance**

An essential property: users **can't combine their keys** to satisfy a policy that neither satisfies alone. In the diagram, User 2 has {student, Monash} and User 3 has {computer science}. Together they have every attribute in the policy, but each user's key components are tied together with randomness, so they **still can't decrypt**.

#### **Attribute-Based Proxy Re-Encryption**

The access policy of an **existing ciphertext** can be changed without decrypting it first. A **proxy** is given a re-encryption key and converts the ciphertext from the old policy to the new one, without ever seeing the message.

| | Policy | Who Can Decrypt |
|---|--------|-----------------|
| Original | {student AND math} | Math students |
| After re-encryption | {student AND computer science} | Computer Science students |

#### **Advantages and Disadvantages**

- **Advantages**:
  - **One-to-many**: one ciphertext can be read by any number of matching users
  - The sender **doesn't need to know who the receivers are**
  - Ciphertext size depends on the **policy**, not on the **number of receivers**
  - Useful for fine-grained access control in large organisations and cloud storage
  - New users who match the policy can decrypt existing ciphertext without re-encrypting it

- **Disadvantages**:
  - **More complex** to manage than the other schemes covered so far
  - **Key escrow**: like a PKG, the authority can generate any key, so it must be fully trusted
  - **Revocation is hard**: removing an attribute from one user (e.g. when they graduate) usually means re-issuing keys or re-encrypting data
  - **Slower** than traditional public-key encryption, since it relies on pairings
  - **Few real deployments** so far. Libraries such as OpenABE exist, but more research is needed for practical use

| Feature | Traditional Public-Key | Identity-Based | Attribute-Based |
|---------|------------------------|----------------|-----------------|
| Encrypt to | One public key | One identity | A policy over attributes |
| Receivers | One | One | Any number of matching users |
| Sender must know receivers? | Yes | Yes | No |
| Certificates needed | Yes | No | No |
| Key issued by | The user | Private Key Generator | Attribute Authority |

In [4]:
# Which users satisfy an attribute-based policy?
# This only shows the ACCESS LOGIC. In real ABE the policy is enforced by the maths of the
# encryption itself, so a user who doesn't match simply cannot decrypt.

users = {
    "User 1": {"student", "computer science", "Monash"},
    "User 2": {"student", "math", "Monash"},
    "User 3": {"lecturer", "computer science", "Monash"},
    "User 4": {"male", "Japanese", "PhD"},
    "User 5": {"female", "Korean", "accountant"},
}

# A policy is a tree of ("AND", ...), ("OR", ...) and attribute names
policies = {
    "CS students at Monash": ("AND", "student", "computer science", "Monash"),
    "Male Japanese or female Korean": ("OR", ("AND", "male", "Japanese"), ("AND", "female", "Korean")),
}


def satisfies(attributes, policy):
    if isinstance(policy, str):
        return policy in attributes
    op, *parts = policy
    results = [satisfies(attributes, p) for p in parts]
    return all(results) if op == "AND" else any(results)


for name, policy in policies.items():
    allowed = [u for u, attrs in users.items() if satisfies(attrs, policy)]
    print(f"{name}: {allowed}")

# Collusion: User 2 (student, Monash) + User 3 (computer science) together match the
# policy, but real ABE ties each user's keys together so they CANNOT combine them.
combined = users["User 2"] | users["User 3"]
print("\nUser 2 + User 3 combined would match:", satisfies(combined, policies["CS students at Monash"]))
print("...but collusion resistance in real ABE stops them pooling their keys.")

CS students at Monash: ['User 1']
Male Japanese or female Korean: ['User 4', 'User 5']

User 2 + User 3 combined would match: True
...but collusion resistance in real ABE stops them pooling their keys.


<br>

***